Check CUDA compiler

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


Write CUDA code to file named kernel.cu

In [13]:
%%writefile kernel_bf16vectorMul.cu
#include <stdio.h>
#include <cuda_bf16.h>

__global__ void bf16_vector_mul(__nv_bfloat16* A, __nv_bfloat16* B, __nv_bfloat16* C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        C[i] = __hmul(A[i], B[i]);
    }
}

int main() {
    const int N = 1 << 20;  // 1M elements
    size_t size = N * sizeof(__nv_bfloat16);

    // Allocate host memory
    __nv_bfloat16* h_A = (__nv_bfloat16*)malloc(size);
    __nv_bfloat16* h_B = (__nv_bfloat16*)malloc(size);
    __nv_bfloat16* h_C = (__nv_bfloat16*)malloc(size);

    // Initialize host vectors
    for (int i = 0; i < N; i++) {
        h_A[i] = __float2bfloat16_rn(1.0f);
        h_B[i] = __float2bfloat16_rn(2.0f);
    }

    // Allocate device memory
    __nv_bfloat16 *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data to device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
    bf16_vector_mul<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    // Copy result back to host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting kernel_bf16vectorMul.cu


Compile ptx (match .cu and .ptx filename)

In [14]:
!nvcc -ptx -arch=sm_80 kernel_bf16vectorMul.cu -o kernel_bf16vectorMul.ptx

Instructions in Vector Add:  

ld.param.dtype  
mov.dtype  
mad.lo.dtype  
mul.wide.dtype  
setp.ge.dtype  
@%p1 bra  
cvta.to.global.dtype (idt this one matters)  
add.dtype